# NBA Game Prediction
Final machine learning pipeline with chronological feature engineering and leakage free model evaluation.

## Stage 1: Load and Prepare Data

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/nba_games.csv")

# Convert game date to datetime
df["gameDate"] = pd.to_datetime(df["gameDateTimeEst"])

# Keep only games from the modern era
df = df[df["gameDate"] >= "2000-01-01"].copy()

# Sort chronologically
df = df.sort_values("gameDate").reset_index(drop=True)

# Target variable
df["HOME_WIN"] = (
    df["homeScore"] > df["awayScore"]
).astype(int)

df = df[[
    "gameId",
    "gameDate",
    "hometeamName",
    "awayteamName",
    "homeScore",
    "awayScore",
    "HOME_WIN"
]].copy()

verify

In [ ]:
df.head()

In [ ]:
df.info()

## Stage 2: Build Chronological Team Timeline

In [ ]:
# Home team perspective
home = df[[
    "gameId",
    "gameDate",
    "hometeamName",
    "homeScore",
    "awayScore"
]].copy()

home.columns = [
    "gameId",
    "gameDate",
    "team",
    "points_scored",
    "points_allowed"
]

home["is_home"] = 1
home["win"] = (home["points_scored"] > home["points_allowed"]).astype(int)


# Away team perspective
away = df[[
    "gameId",
    "gameDate",
    "awayteamName",
    "awayScore",
    "homeScore"
]].copy()

away.columns = [
    "gameId",
    "gameDate",
    "team",
    "points_scored",
    "points_allowed"
]

away["is_home"] = 0
away["win"] = (away["points_scored"] > away["points_allowed"]).astype(int)


# Combine into single timeline
team_timeline = pd.concat([home, away], ignore_index=True)

# Sort chronologically within each team
team_timeline = team_timeline.sort_values(["team", "gameDate"]).reset_index(drop=True)

# Point differential
team_timeline["point_diff"] = (
    team_timeline["points_scored"] - team_timeline["points_allowed"]
)

verify

In [ ]:
team_timeline.head()

In [ ]:
team_timeline.columns

In [ ]:
len(df), len(team_timeline)

## Stage 3: Create Leakage Free Features

In [ ]:
team_timeline = team_timeline.sort_values(["team", "gameDate"]).copy()

team_timeline["win_5"] = (
    team_timeline.groupby("team")["win"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
)

team_timeline["win_10"] = (
    team_timeline.groupby("team")["win"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
)

team_timeline["pd_5"] = (
    team_timeline.groupby("team")["point_diff"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
)

team_timeline["pd_10"] = (
    team_timeline.groupby("team")["point_diff"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
)

verify

In [ ]:
team_timeline.columns

In [ ]:
"team" in team_timeline.columns

In [ ]:
team_timeline.head(10)

## Stage 4: Build Machine Learning Dataset

1. Base game table

In [ ]:
ml_base = df[[
    "gameId",
    "gameDate",
    "hometeamName",
    "awayteamName",
    "homeScore",
    "awayScore",
    "HOME_WIN"
]].copy()

2. Prepare feature table (shared source)

In [ ]:
features = team_timeline[[
    "gameId",
    "team",
    "win_5",
    "win_10",
    "pd_5",
    "pd_10"
]].copy()

3. Home features merge

In [ ]:
home_features = features.rename(columns={
    "team": "hometeamName",
    "win_5": "home_win_5",
    "win_10": "home_win_10",
    "pd_5": "home_pd_5",
    "pd_10": "home_pd_10"
})

ml_base = ml_base.merge(
    home_features,
    on=["gameId", "hometeamName"],
    how="left"
)

4. Away features merge

In [ ]:
away_features = features.rename(columns={
    "team": "awayteamName",
    "win_5": "away_win_5",
    "win_10": "away_win_10",
    "pd_5": "away_pd_5",
    "pd_10": "away_pd_10"
})

ml_base = ml_base.merge(
    away_features,
    on=["gameId", "awayteamName"],
    how="left"
)

5. Clean final dataset

In [ ]:
ml_base = ml_base.dropna().reset_index(drop=True)

ml_base.head()